In [1]:
import torch
from torch import nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.attn_w = None

    def _rotate_half(self, x):
        # x: (batch, num_heads, seq_len, d_k)
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)

    def _rope(self, q, k, base=10000):
        # q, k: (batch, num_heads, seq_len, d_k)
        seq_len = q.size(-2)
        freqs = torch.arange(0, self.d_k, 2, device=q.device) / self.d_k
        freqs = base ** freqs                                           # (d_k/2,)
        angles = torch.arange(seq_len, device=q.device)[:, None] / freqs[None, :]  # (seq_len, d_k/2)
        sin = angles.sin().repeat_interleave(2, dim=-1)[None, None]   # (1, 1, seq_len, d_k)
        cos = angles.cos().repeat_interleave(2, dim=-1)[None, None]   # (1, 1, seq_len, d_k)
        return q * cos + self._rotate_half(q) * sin, \
            k * cos + self._rotate_half(k) * sin
    
    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        """
        batch_size = x.size(0)
        Q = self.w_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        Q, K = self._rope(Q, K)
        score = Q @ K.transpose(-2, -1) / self.d_k ** 0.5
        attn_w = torch.softmax(score, dim=-1)
        self.attn_w = attn_w.detach()
        output = attn_w @ V # (batch, num_heads, seq_len, d_k)
        output = output.transpose(1, 2).reshape(batch_size, -1, self.d_model)
        return self.out(output) # (batch, seq_len, d_model)
    
class TransformerEncoderBlock(nn.Module):
    def __init__(self, ffn_dim, d_model, num_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(),
            nn.Linear(ffn_dim, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        x_norm = self.norm1(x)
        x = x + self.attn(x_norm)
        x_norm = self.norm2(x)
        x = x + self.ffn(x_norm)
        return x

class TransformerEncoder(nn.Module): # 新增encoder
    def __init__(self, vocab_size, output_dim, ffn_dim, d_model, num_heads, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(ffn_dim, d_model, num_heads)
            for _ in range(num_layers)
        ])
        self.out = nn.Linear(d_model, output_dim)
    
    def forward(self, x):
        x = self.embedding(x)  # (batch, seq_len, d_model)
        for layer in self.layers:
            x = layer(x)
        return self.out(x)

In [39]:
def get_sort_batch(batch_size, seq_len, vocab_size):
    """
    排序任务数据集
    输入: 乱序的整数序列      e.g. [3, 1, 4, 1, 5, 2]
    标签: 升序排列后的序列    e.g. [1, 1, 2, 3, 4, 5]
    """
    x = torch.randint(0, vocab_size, (batch_size, seq_len), dtype=torch.long)
    y = x.sort(dim=-1).values  # (batch_size, seq_len), long
    return x, y

epochs = 1000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = TransformerEncoder(
    vocab_size=10,
    d_model=128,
    ffn_dim=256,
    num_heads=4,
    num_layers=3,
    output_dim=10
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss().to(device)

model.train()
for i in range(epochs):
    train_x, train_y = get_sort_batch(1024, 10, 10)
    train_x, train_y = train_x.to(device), train_y.to(device)
    optimizer.zero_grad()
    pred = model(train_x)                      # (batch, seq_len, vocab_size)
    loss = criterion(pred.transpose(1, 2), train_y)
    loss.backward()
    optimizer.step()
    if (i+1) % 100 == 0:
        print(f"Epoch {i+1}, Loss: {loss.item():.4f}")

Using device: cuda
Epoch 100, Loss: 0.0899
Epoch 200, Loss: 0.0052
Epoch 300, Loss: 0.9875
Epoch 400, Loss: 0.4743
Epoch 500, Loss: 0.0861
Epoch 600, Loss: 0.0111
Epoch 700, Loss: 0.0036
Epoch 800, Loss: 0.0037
Epoch 900, Loss: 0.0025
Epoch 1000, Loss: 0.0023


In [ ]:
test_x, test_y = get_sort_batch(4, 6, 10)
test_x, test_y = test_x.to(device), test_y.to(device)
model.eval()
with torch.no_grad():
    pred = model(test_x)
    pred_tokens = pred.argmax(dim=-1)   # (batch, seq_len)

    for i in range(test_x.size(0)):
        print(f"input : {test_x[i].tolist()}")
        print(f"pred  : {pred_tokens[i].tolist()}")
        print(f"target: {test_y[i].tolist()}")
        print("-" * 30)


input : [2, 1, 7, 3, 4, 1]
pred  : [1, 1, 2, 3, 3, 3]
target: [1, 1, 2, 3, 4, 7]
------------------------------
input : [7, 2, 9, 3, 6, 9]
pred  : [2, 3, 6, 7, 9, 9]
target: [2, 3, 6, 7, 9, 9]
------------------------------
input : [6, 5, 9, 8, 2, 8]
pred  : [2, 5, 6, 8, 8, 8]
target: [2, 5, 6, 8, 8, 9]
------------------------------
input : [3, 0, 7, 1, 7, 8]
pred  : [0, 1, 3, 7, 7, 8]
target: [0, 1, 3, 7, 7, 8]
------------------------------
